# Running in VSCode:

1. Set kernal to python-interface-to-workflows 3.11.x
2. Hit F1, run Jupyter: Import Notebook to Script
3. Click notebook_division.ipynb
4. Run the cells sequentially

In [ ]:
import json

from hera.workflows import (
    Artifact,
    EmptyDirVolume,
    Steps,
    Workflow,
    script,  # pyright: ignore[reportUnknownVariableType]
)
from hera.workflows import models as m


@script(
    volume_mounts=[m.VolumeMount(name="output-dir", mount_path="/output-dir/")],
    outputs=Artifact(name="json-output", path="/output-dir/output.json"),
)
def do_division(a: int, b: int):
    div = a / b
    intdiv = a // b
    remain = a % b
    dictionary_of_results = {
        "divide": div,
        "quotient": intdiv,
        "remainder": remain,
    }
    with open("/output-dir/output.json", "w") as otpt:
        json.dump(dictionary_of_results, otpt)


with Workflow(
    generate_name="hera-division-",  # when running on graphql this should be name
    entrypoint="divide",
    api_version="argoproj.io/v1alpha1",
    kind="Workflow",  # ClusterWorkflowTemplate", when on graphql
    labels={"workflows.diamond.ac.uk/science-group-examples": "true"},
    annotations={
        "workflows.argoproj.io/title": "Division via hera test",
        "workflows.argoproj.io/description": """Takes a numerical input and returns
    the remainder, output float, and output string to a json file""",
        "workflows.diamond.ac.uk/repository": "https://github.com/DiamondLightSource/python-interface-to-workflows",
    },
    volumes=EmptyDirVolume(name="output-dir", mount_path="/output-dir"),
) as w:
    with Steps(name="divide"):
        do_division(name="first", arguments={"a": 2, "b": 5})



In [ ]:
with open("division_from_jupyter.yaml", "w") as div:
    div.write(w.to_yaml())  # pyright: ignore[reportUnknownMemberType]

In [ ]:
from python_interface_to_workflows.submit_to_argo import submit_workflow_to_argo

submit_workflow_to_argo(w)

expiry_str: '1784802224' 1784801151.7235122


ValueError: invalid literal for int() with base 10: "'1784802224'"